# TradeSense · Component 03 · notebook 52: does the frozen engine respond to REAL BTC / ETH market stress? (protocol D1; novelty 2 Part D; CPU, minutes)

**Question.** On the real Aug-Oct 2024 BTCUSDT / ETHUSDT bars, do the frozen fusion alarms (Watch / Alert) and the UNSEEN PATTERN signal respond to real abnormal market events (the 1% most volatile hours, plus three publicly reported anchor days), compared with ordinary hours?

**Honest framing.** These are real market-stress events, **not** confirmed manipulation (no such public labels exist for BTC/ETH). A price-only baseline shows how much the engine adds beyond plain volatility. No model, feature, normalisation or threshold is changed; this is a new pre-registered analysis on saved TEST-B scores.

**Protocol text hashed by this notebook:**


```
# Protocol D1 v1 - real BTC/ETH market-stress check of the frozen engine and the UNSEEN PATTERN signal (novelty 2, Part D)
Written BEFORE any score is read against these events. Nothing frozen is changed (models, 6 features, normalisation, fusion stacker, thresholds q). Uses saved TEST-B scores (library sha 3b5d9df23521adf9) on the Aug 1 - Oct 31 2024 genuine bars; a new pre-registered analysis (TEST-B was opened once in T0).

## What this is and is not
Real BTCUSDT / ETHUSDT stress events. There is no public, confirmed-manipulation label for BTC/ETH, so these are REAL ABNORMAL MARKET EVENTS, not proof of manipulation. A statement of the form "the engine responds to real market stress" is the only claim allowed. The July anchor (5 Jul, Mt. Gox) lies in the development period used to fix q, so it is shown descriptively only and is not tested.

## Real-event definition (price only, no engine score used)
1. EXTREME HOUR: a UTC hour block (60 bars) whose realised volatility sqrt(sum of squared 1-minute log returns) is in the top 1% of that symbol's hours over Aug 1 - Oct 31 2024. Control hours: all other hours except those within 2 hours of an extreme hour.
2. ANCHOR DAYS (publicly reported, day level, UTC): 2024-08-05 (global sell-off, BTC low about 49.1k, over 1 billion USD liquidations; Benzinga 5 Aug 2024), 2024-09-06 (weak US jobs report, BTC low 53.8k; Bitcoinist 7 Sep 2024), 2024-09-18 (FOMC rate decision, scheduled). Sources are recorded in the project notes.

## Engine readings (declared now)
Alert bar = fusion above its rolling 7-day alert line at budget 2/day (declared q). Watch bar = fusion above the Watch line (declared_watch_v1.json, budget 4). Hour flagged = at least one Alert-or-Watch bar in the hour. UNSEEN hour = at least one bar with AE alarm (budget 2) and neither XGBoost nor Transformer alarm (budget 2 each). Both-symbols hour = flagged for BTC and ETH in the same hour.

## Hypotheses
- H-D1: lift = P(flagged | extreme hour) / P(flagged | control hour), mean of BTC and ETH, is >= 2.0 and the day-block bootstrap 95% lower bound is > 1.
- H-D2: on at least 2 of the 3 anchor days, the number of Alert episodes (gap 5) of both symbols is above the 90th percentile of that symbol's other days.
- H-D3: lift of "both symbols flagged in the same hour" (extreme versus control) is >= 2.0 with the lower bound > 1.
- Descriptive only: UNSEEN share among flagged extreme hours versus flagged control hours; engine lift versus a simple price-only baseline (per-bar |log return| above its own rolling 7-day quantile, matched to the same genuine alarm hours); lift under circularly shifted alarms (chance); per-month split; table of the 10 most extreme hours per symbol.

## Known weaknesses, stated in advance
Few events (about 22 extreme hours per symbol, 3 anchor days). A price-volatility event is partly what the features measure, so a lift above 1 is expected and is not evidence of manipulation detection; the price-only baseline shows how much beyond plain volatility the engine adds. nb44 already found that the fusion score did not mark 5 Aug as an unusual day; H-D2 may fail on that day. Failed hypotheses are reported as failed.

```

In [ ]:
import os, sys, json, hashlib, warnings, glob
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'; SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 300 if MOCK else 3000; NSHIFT = 100 if MOCK else 500
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'protocol B2 missing'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
PROTO_D1 = r"""# Protocol D1 v1 - real BTC/ETH market-stress check of the frozen engine and the UNSEEN PATTERN signal (novelty 2, Part D)
Written BEFORE any score is read against these events. Nothing frozen is changed (models, 6 features, normalisation, fusion stacker, thresholds q). Uses saved TEST-B scores (library sha 3b5d9df23521adf9) on the Aug 1 - Oct 31 2024 genuine bars; a new pre-registered analysis (TEST-B was opened once in T0).

## What this is and is not
Real BTCUSDT / ETHUSDT stress events. There is no public, confirmed-manipulation label for BTC/ETH, so these are REAL ABNORMAL MARKET EVENTS, not proof of manipulation. A statement of the form "the engine responds to real market stress" is the only claim allowed. The July anchor (5 Jul, Mt. Gox) lies in the development period used to fix q, so it is shown descriptively only and is not tested.

## Real-event definition (price only, no engine score used)
1. EXTREME HOUR: a UTC hour block (60 bars) whose realised volatility sqrt(sum of squared 1-minute log returns) is in the top 1% of that symbol's hours over Aug 1 - Oct 31 2024. Control hours: all other hours except those within 2 hours of an extreme hour.
2. ANCHOR DAYS (publicly reported, day level, UTC): 2024-08-05 (global sell-off, BTC low about 49.1k, over 1 billion USD liquidations; Benzinga 5 Aug 2024), 2024-09-06 (weak US jobs report, BTC low 53.8k; Bitcoinist 7 Sep 2024), 2024-09-18 (FOMC rate decision, scheduled). Sources are recorded in the project notes.

## Engine readings (declared now)
Alert bar = fusion above its rolling 7-day alert line at budget 2/day (declared q). Watch bar = fusion above the Watch line (declared_watch_v1.json, budget 4). Hour flagged = at least one Alert-or-Watch bar in the hour. UNSEEN hour = at least one bar with AE alarm (budget 2) and neither XGBoost nor Transformer alarm (budget 2 each). Both-symbols hour = flagged for BTC and ETH in the same hour.

## Hypotheses
- H-D1: lift = P(flagged | extreme hour) / P(flagged | control hour), mean of BTC and ETH, is >= 2.0 and the day-block bootstrap 95% lower bound is > 1.
- H-D2: on at least 2 of the 3 anchor days, the number of Alert episodes (gap 5) of both symbols is above the 90th percentile of that symbol's other days.
- H-D3: lift of "both symbols flagged in the same hour" (extreme versus control) is >= 2.0 with the lower bound > 1.
- Descriptive only: UNSEEN share among flagged extreme hours versus flagged control hours; engine lift versus a simple price-only baseline (per-bar |log return| above its own rolling 7-day quantile, matched to the same genuine alarm hours); lift under circularly shifted alarms (chance); per-month split; table of the 10 most extreme hours per symbol.

## Known weaknesses, stated in advance
Few events (about 22 extreme hours per symbol, 3 anchor days). A price-volatility event is partly what the features measure, so a lift above 1 is expected and is not evidence of manipulation detection; the price-only baseline shows how much beyond plain volatility the engine adds. nb44 already found that the fusion score did not mark 5 Aug as an unusual day; H-D2 may fail on that day. Failed hypotheses are reported as failed.
"""
PHD = hashlib.sha256(PROTO_D1.encode()).hexdigest(); pp = OUT/'protocol_d1_v1.md'
if pp.exists(): assert pp.read_text() == PROTO_D1, 'protocol D1 exists and differs from this notebook'
else: pp.write_text(PROTO_D1)
(OUT/'protocol_d1_v1.sha256').write_text(PHD + '  protocol_d1_v1.md\n')
for f in ['test_b_results_v1.json', 'declared_final_set_v1.json', 'declared_ae_v2.json', 'declared_fusion_v0.json', 'declared_watch_v1.json', 'b2_lib.py', 'c03_infer.py']: assert (OUT/f).exists(), f'{f} missing'
T32 = json.loads((OUT/'test_b_results_v1.json').read_text())
for s in SYMBOLS: assert sha_file(OUT/'scores'/f'test_b_{s}.npz') == T32['test_score_files_sha16'][s], 'saved TEST scores differ from what notebook 32 recorded'
assert sha_file(OUT/'c03_infer.py') == T32['library_sha16'] and sha_file(OUT/'b2_lib.py') == T32['b2_lib_sha16'], 'library files differ from the ones that opened TEST'
sys.path.insert(0, str(OUT)); import b2_lib as B; import c03_infer as c
say('protocol D1 v1 sha256 ' + PHD[:16] + '; protocol B2 intact ' + ph[:16] + '; library ' + T32['library_sha16'] + ('  [MOCK RUN]' if MOCK else ''))

Mounted at /content/drive
protocol D1 v1 sha256 bc55b97af9614302; protocol B2 intact 443f99d5faf5929f; library 3b5d9df23521adf9


## 1 · Saved scores and engine readings (Aug 1 - Oct 31 2024)

In [ ]:
AOFULL = 132480; NH = AOFULL // 60; NDAYS = AOFULL // 1440
DF = json.load(open(OUT/'declared_final_set_v1.json')); AE2 = json.load(open(OUT/'declared_ae_v2.json')); FUSD = json.load(open(OUT/'declared_fusion_v0.json')); DW = json.load(open(OUT/'declared_watch_v1.json'))
MODELS = c.MODEL_NAMES; TAG = dict(zip(MODELS, ['AE', 'XGB', 'TR']))
QM = {MODELS[0]: AE2['q_T1_by_budget']['2'], MODELS[1]: DF['models'][MODELS[1]]['q_T1_by_budget']['2'], MODELS[2]: DF['models'][MODELS[2]]['q_T1_by_budget']['2']}
QA = FUSD['q_T1_by_budget']['2']; QW = DW['watch']['q']['4']
def find1(pat):
    cc = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert cc, pat; return sorted(cc)[0]
R = {}
for s in SYMBOLS:
    b = pd.read_parquet(BASE/'layer1_aug_oct_v1'/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet'); assert len(b) == AOFULL
    z = np.load(OUT/'scores'/f'test_b_{s}.npz'); F = z['FUS__GEN'].astype('float64'); F[:B.WARM] = np.nan
    sc = {m: z[TAG[m] + '__GEN'].astype('float64') for m in MODELS}
    for m in MODELS: sc[m][:B.WARM] = np.nan
    alert = F > c.rolling_threshold(F, QA[s]); watch = (F > c.rolling_threshold(F, QW[s])) | alert
    ae = sc[MODELS[0]] > c.rolling_threshold(sc[MODELS[0]], QM[MODELS[0]][s]); xg = sc[MODELS[1]] > c.rolling_threshold(sc[MODELS[1]], QM[MODELS[1]][s]); tr = sc[MODELS[2]] > c.rolling_threshold(sc[MODELS[2]], QM[MODELS[2]][s])
    unseen = ae & ~xg & ~tr; lr = b['log_returns'].to_numpy(float); lr = np.nan_to_num(lr)
    # price-only baseline alarm: |log return| above its own rolling 7-day quantile, quantile chosen so the genuine hourly-flag rate matches the engine's Watch rate
    ab = np.abs(lr); ts = b.index
    R[s] = dict(alert=alert, watch=watch, unseen=unseen, lr=lr, ab=ab, ts=ts, F=F)
    say(f'{s}: bars {AOFULL}, hours {NH}, Alert bars {int(alert.sum())}, Watch-or-Alert bars {int(watch.sum())}, UNSEEN bars {int(unseen.sum())}')

BTCUSDT: bars 132480, hours 2208, Alert bars 431, Watch-or-Alert bars 954, UNSEEN bars 948
ETHUSDT: bars 132480, hours 2208, Alert bars 439, Watch-or-Alert bars 709, UNSEEN bars 1222


## 2 · Real-event definition (price only) and price-only baseline

In [ ]:
def hourly(x, how):
    a = np.asarray(x)[:NH * 60].reshape(NH, 60); return {'any': a.any(axis=1), 'rv': np.sqrt((a ** 2).sum(axis=1))}[how]
EXT = {}; CTRL = {}; DAYH = (np.arange(NH) * 60) // 1440
for s in SYMBOLS:
    rv = hourly(R[s]['lr'], 'rv'); cut = np.quantile(rv, 0.99); ext = rv >= cut
    near = np.convolve(ext.astype(int), np.ones(5, int), mode='same') > 0; EXT[s] = ext; CTRL[s] = ~near
    R[s].update(rv=rv, flag=hourly(R[s]['watch'], 'any'), flagA=hourly(R[s]['alert'], 'any'), unh=hourly(R[s]['unseen'], 'any'))
    say(f'{s}: extreme hours (top 1% realised volatility) {int(ext.sum())}, control hours {int(CTRL[s].sum())}')
# price-only baseline: per-bar |log return| over its rolling 7-day quantile; q picked so its flagged-hour rate over the whole period equals the engine Watch flagged-hour rate (rate matching uses no event information)
for s in SYMBOLS:
    target = R[s]['flag'].mean(); lo, hi = 0.9, 0.99999
    for _ in range(18):
        qm = (lo + hi) / 2; fl = hourly(R[s]['ab'] > c.rolling_threshold(np.where(np.arange(AOFULL) < B.WARM, np.nan, R[s]['ab']), qm), 'any').mean()
        if fl > target: lo = qm
        else: hi = qm
    R[s]['pflag'] = hourly(R[s]['ab'] > c.rolling_threshold(np.where(np.arange(AOFULL) < B.WARM, np.nan, R[s]['ab']), hi), 'any'); R[s]['pq'] = hi
DAYS = pd.Series(R[SYMBOLS[0]]['ts']).dt.strftime('%Y-%m-%d').to_numpy()[::1440][:NDAYS]
ANCH = ['2024-08-05', '2024-09-06', '2024-09-18']
say('anchor days present in the data: ' + str({d: (d in set(DAYS)) for d in ANCH}))

BTCUSDT: extreme hours (top 1% realised volatility) 23, control hours 2141
ETHUSDT: extreme hours (top 1% realised volatility) 23, control hours 2136
anchor days present in the data: {'2024-08-05': True, '2024-09-06': True, '2024-09-18': True}


## 3 · Unit tests

In [ ]:
# T01 hourly aggregation equals a plain loop
x = np.random.default_rng(0).random(NH * 60 + 7) < 0.002; h = hourly(x, 'any'); assert all(h[i] == x[i * 60:(i + 1) * 60].any() for i in range(NH))
rr = np.random.default_rng(1).normal(size=NH * 60); assert np.allclose(hourly(rr, 'rv')[5], np.sqrt((rr[300:360] ** 2).sum())); say('PASS T01 hourly flag and realised volatility equal a plain loop')
# T02 control hours never lie within two hours of an extreme hour; extreme count is ~1%
for s in SYMBOLS:
    e = np.flatnonzero(EXT[s]); assert all(not CTRL[s][max(0, i - 2):i + 3].any() for i in e); assert 0.005 <= EXT[s].mean() <= 0.02
say('PASS T02 control hours are at least 3 hours away from every extreme hour; about 1% of hours are extreme')
# T03 event definition does not use any engine score: change the scores, extreme hours stay identical
s = SYMBOLS[0]; alt = np.abs(hourly(R[s]['lr'], 'rv')) >= np.quantile(hourly(R[s]['lr'], 'rv'), 0.99); assert (alt == EXT[s]).all(); say('PASS T03 extreme hours depend on price only (no engine score is used)')
# T04 the lift function equals a hand calculation
def lift(flag, ext, ctrl):
    return float(flag[ext].mean() / max(flag[ctrl].mean(), 1e-12))
f_ = np.array([1, 1, 0, 0, 1, 0, 0, 0], bool); e_ = np.array([1, 1, 0, 0, 0, 0, 0, 0], bool); k_ = np.array([0, 0, 0, 0, 0, 1, 1, 1], bool)
f2 = np.array([1, 0, 1, 0, 0, 0, 0, 0], bool); assert abs(lift(f2, e_, np.array([0, 0, 1, 1, 1, 1, 0, 0], bool)) - 0.5 / 0.25) < 1e-12; say('PASS T04 lift equals a hand calculation (0.5 / 0.25 = 2)')

PASS T01 hourly flag and realised volatility equal a plain loop
PASS T02 control hours are at least 3 hours away from every extreme hour; about 1% of hours are extreme
PASS T03 extreme hours depend on price only (no engine score is used)
PASS T04 lift equals a hand calculation (0.5 / 0.25 = 2)


## 4 · Results (protocol D1) and save

In [ ]:
def mean_lift(flagkey, mask_h=None):
    return float(np.mean([lift(R[s][flagkey], EXT[s], CTRL[s]) for s in SYMBOLS]))
def boot_lift(fn, n=NBOOT, seed=0):
    rg = np.random.default_rng(seed); ud = np.arange(NDAYS); ix = [np.flatnonzero(DAYH == d) for d in ud]; out = []
    for _ in range(n):
        h = np.concatenate([ix[d] for d in rg.choice(ud, NDAYS)]); out.append(fn(h))
    return np.nanpercentile(out, [2.5, 97.5])
def lift_sub(flagget, h): return float(np.mean([flagget(s)[h][EXT[s][h]].mean() / max(flagget(s)[h][CTRL[s][h]].mean(), 1e-12) if EXT[s][h].any() and CTRL[s][h].any() else np.nan for s in SYMBOLS]))
both = hourly(R[SYMBOLS[0]]['watch'], 'any') & hourly(R[SYMBOLS[1]]['watch'], 'any')
EXTU = EXT[SYMBOLS[0]] | EXT[SYMBOLS[1]]; CT2 = CTRL[SYMBOLS[0]] & CTRL[SYMBOLS[1]]
rows = []
for s in SYMBOLS:
    rows.append({'symbol': s, 'extreme hours': int(EXT[s].sum()), 'P(flagged | extreme)': R[s]['flag'][EXT[s]].mean(), 'P(flagged | control)': R[s]['flag'][CTRL[s]].mean(), 'P(Alert | extreme)': R[s]['flagA'][EXT[s]].mean(), 'P(Alert | control)': R[s]['flagA'][CTRL[s]].mean(),
                 'price-only baseline P(flag | extreme)': R[s]['pflag'][EXT[s]].mean(), 'price-only baseline P(flag | control)': R[s]['pflag'][CTRL[s]].mean(), 'UNSEEN share of flagged extreme hours': float(R[s]['unh'][EXT[s] & R[s]['flag']].mean()) if (EXT[s] & R[s]['flag']).any() else np.nan, 'UNSEEN share of flagged control hours': float(R[s]['unh'][CTRL[s] & R[s]['flag']].mean())})
T = pd.DataFrame(rows); T.to_csv(OUT/'d1_event_table.csv', index=False); say('--- extreme-hour table (flagged = any Watch-or-Alert bar in the hour) ---'); say(T.round(3).to_string(index=False))
L1 = mean_lift('flag'); ci1 = boot_lift(lambda h: lift_sub(lambda s: R[s]['flag'], h)); LP = mean_lift('pflag'); ciP = boot_lift(lambda h: lift_sub(lambda s: R[s]['pflag'], h))
LB = float(both[EXTU].mean() / max(both[CT2].mean(), 1e-12)); ciB = boot_lift(lambda h: float(both[h][EXTU[h]].mean() / max(both[h][CT2[h]].mean(), 1e-12)) if EXTU[h].any() and CT2[h].any() else np.nan)
# chance: shift both symbols' flags by a random whole number of days
rg = np.random.default_rng(5); ch = []
for _ in range(NSHIFT):
    k = int(rg.integers(3, NDAYS - 3)) * 24; ch.append(np.mean([lift(np.roll(R[s]['flag'], k), EXT[s], CTRL[s]) for s in SYMBOLS]))
ch = np.nanpercentile(ch, [2.5, 50, 97.5])
say(f'engine lift (flagged, extreme / control): {L1:.2f} (day-block 95% {ci1[0]:.2f}..{ci1[1]:.2f}); price-only baseline lift {LP:.2f} ({ciP[0]:.2f}..{ciP[1]:.2f}); chance by shifting the flags {ch[0]:.2f}..{ch[2]:.2f} (median {ch[1]:.2f})')
say(f'both symbols flagged in the same hour: lift {LB:.2f} ({ciB[0]:.2f}..{ciB[1]:.2f})')
# anchor days: Alert episodes per day versus the other days
def eps_per_day(s):
    t = np.flatnonzero(R[s]['alert']); e = np.split(t, np.flatnonzero(np.diff(t) > 5) + 1) if len(t) else []; st = np.array([x[0] for x in e], int); return np.bincount(st // 1440, minlength=NDAYS)
EP = {s: eps_per_day(s) for s in SYMBOLS}; ARow = []; okd = 0
for d in ANCH:
    if d not in set(DAYS): continue
    i = int(np.flatnonzero(DAYS == d)[0]); rr = {'day': d}; both_above = True
    for s in SYMBOLS:
        others = np.delete(EP[s], i); p90 = np.percentile(others, 90); rr[f'{s} Alert episodes'] = int(EP[s][i]); rr[f'{s} p90 other days'] = float(p90); both_above &= bool(EP[s][i] > p90)
    rr['both above p90'] = both_above; okd += int(both_above); ARow.append(rr)
AT = pd.DataFrame(ARow); AT.to_csv(OUT/'d1_anchor_table.csv', index=False); say(''); say('--- anchor days ---'); say(AT.round(2).to_string(index=False))
top = []
for s in SYMBOLS:
    for i in np.argsort(-R[s]['rv'])[:10]: top.append({'symbol': s, 'hour (UTC)': str(R[s]['ts'][i * 60])[:16], 'realised vol': float(R[s]['rv'][i]), 'Watch/Alert': bool(R[s]['flag'][i]), 'Alert': bool(R[s]['flagA'][i]), 'UNSEEN': bool(R[s]['unh'][i])})
TT = pd.DataFrame(top); TT.to_csv(OUT/'d1_top_hours.csv', index=False); say(''); say('--- 10 most extreme hours per symbol ---'); say(TT.round(5).to_string(index=False))
HD1 = bool(L1 >= 2.0 and ci1[0] > 1); HD2 = bool(okd >= 2); HD3 = bool(LB >= 2.0 and ciB[0] > 1)
say(''); say('=== pre-registered hypotheses (protocol D1 v1; new analysis on saved TEST-B scores) ===')
say(f'H-D1 flagged-hour lift >= 2.0 with interval above 1: {L1:.2f} ({ci1[0]:.2f}..{ci1[1]:.2f}) -> {"HOLDS" if HD1 else "FAILS"}')
say(f'H-D2 both symbols above their p90 Alert-episode day on >= 2 of the 3 anchor days: {okd} of {len(AT)} -> {"HOLDS" if HD2 else "FAILS"}')
say(f'H-D3 both-symbols-flagged lift >= 2.0 with interval above 1: {LB:.2f} ({ciB[0]:.2f}..{ciB[1]:.2f}) -> {"HOLDS" if HD3 else "FAILS"}')
HYP = {'H-D1': HD1, 'H-D2': HD2, 'H-D3': HD3}; say(f'{sum(HYP.values())} of 3 hold. Failed hypotheses are reported as failed; nothing was retuned.')
say('These are real market-stress events, not confirmed manipulation. Compare the engine lift with the price-only baseline lift: only the part above plain volatility is specific to the engine.')
RES = {'protocol_d1_v1_sha256': PHD, 'hypotheses': HYP, 'lift_engine': L1, 'lift_ci': [float(x) for x in ci1], 'lift_price_only': LP, 'lift_price_only_ci': [float(x) for x in ciP], 'lift_chance': [float(x) for x in ch], 'lift_both': LB, 'lift_both_ci': [float(x) for x in ciB], 'anchor_days_ok': okd, 'mock': MOCK}
(OUT/'d1_results.json').write_text(json.dumps(RES, indent=1, default=str)); (OUT/'d1_share_block.txt').write_text('\n'.join(LINES))
say('Saved: d1_results.json, d1_event_table.csv, d1_anchor_table.csv, d1_top_hours.csv, d1_share_block.txt in models_b2_v1/ . Send the share block to Claude.')

--- extreme-hour table (flagged = any Watch-or-Alert bar in the hour) ---
 symbol  extreme hours  P(flagged | extreme)  P(flagged | control)  P(Alert | extreme)  P(Alert | control)  price-only baseline P(flag | extreme)  price-only baseline P(flag | control)  UNSEEN share of flagged extreme hours  UNSEEN share of flagged control hours
BTCUSDT             23                 0.087                 0.149               0.087               0.078                                  0.826                                  0.139                                    0.5                                  0.144
ETHUSDT             23                 0.087                 0.112               0.043               0.067                                  0.739                                  0.104                                    0.0                                  0.192
engine lift (flagged, extreme / control): 0.68 (day-block 95% 0.20..1.88); price-only baseline lift 6.52 (5.34..9.08); chance by shifting